# 面试问题：LLM checkpoint 怎样用 manifest、分片和兼容性门禁安全加载？

**一句话回答。** 加载大模型不是遍历目录后把文件交给框架。先冻结 base model、config、tokenizer、chat template、dtype 和 tensor-parallel 语义；再以不可变 manifest 指明每个参数所属 shard、形状、摘要和 revision。只有完整、兼容且未被篡改的 manifest 能进入加载计划，失败时应拒绝而不是静默随机初始化。

本 Notebook 只以受控小数据实现数据合同、状态机和断言，不调用大模型、真实 OAuth、真实文件或外部工具。断言验证机制不代表生产性能、安全或合规结论。

**资料入口。** [Hugging Face 大模型分片 checkpoint 文档](https://huggingface.co/docs/transformers/main/big_models) 说明 index 将参数名映射到各 shard；本例只模拟元数据合同，不读取真实权重。

In [ ]:
question = "LLM checkpoint manifest 与分片兼容性"  # 执行本行的状态、计算或校验逻辑。
assert "manifest" in question  # 执行本行的状态、计算或校验逻辑。
assert 2 * 3 == 6  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 制品边界

分片解决的是峰值加载内存与分发问题，不自动保证正确性。一个 checkpoint bundle 至少需要模型配置、tokenizer、generation 配置、权重索引、shard 清单和发布摘要；这些对象要以同一个 bundle revision 绑定，避免把 A 模型的 tokenizer 或 B 模型的 adapter 混入。

In [ ]:
bundle = {"revision": "model-r7", "base": "tiny-1", "tokenizer": "tok-r3", "template": "chat-r2", "dtype": "bf16"}  # 执行本行的状态、计算或校验逻辑。
expected_shapes = {"embed.weight": (8, 4), "layer.0.weight": (4, 4), "lm_head.weight": (8, 4)}  # 执行本行的状态、计算或校验逻辑。
assert bundle["revision"] == "model-r7"  # 执行本行的状态、计算或校验逻辑。
assert bundle["tokenizer"] == "tok-r3"  # 执行本行的状态、计算或校验逻辑。
assert len(expected_shapes) == 3  # 执行本行的状态、计算或校验逻辑。

## 2. 索引不变量

index 中每个 tensor key 必须恰好出现一次，引用的 shard 必须存在，期望 tensor 集必须无缺失、无未知项。生产还会验证字节摘要、safetensors 元数据和解析器版本；教学代码以字符串摘要代替真实哈希，目的在于展示拒绝路径。

In [ ]:
shards = {"weights-01.safetensors": {"sha": "aa11", "keys": ("embed.weight", "layer.0.weight")}, "weights-02.safetensors": {"sha": "bb22", "keys": ("lm_head.weight",)}}  # 执行本行的状态、计算或校验逻辑。
index = {"embed.weight": "weights-01.safetensors", "layer.0.weight": "weights-01.safetensors", "lm_head.weight": "weights-02.safetensors"}  # 执行本行的状态、计算或校验逻辑。
assert set(index) == set(expected_shapes)  # 执行本行的状态、计算或校验逻辑。
assert set(index.values()) == set(shards)  # 执行本行的状态、计算或校验逻辑。
assert sum(len(item["keys"]) for item in shards.values()) == 3  # 执行本行的状态、计算或校验逻辑。

## 3. 形状与语义

即便文件可读，tensor shape、层数、vocab size 和 tied-weight 关系也可能不匹配。严格加载应把缺失 key、意外 key、shape mismatch 变成可审计错误；允许部分加载只能由显式迁移策略驱动，绝不能在服务请求路径上悄悄发生。

In [ ]:
def validate_index(index_value, shards_value, expected_value):  # 执行本行的状态、计算或校验逻辑。
    listed = [key for shard in shards_value.values() for key in shard["keys"]]  # 执行本行的状态、计算或校验逻辑。
    return set(index_value) == set(expected_value) and set(index_value.values()) == set(shards_value) and len(listed) == len(set(listed)) == len(index_value)  # 执行本行的状态、计算或校验逻辑。
validated = validate_index(index, shards, expected_shapes)  # 执行本行的状态、计算或校验逻辑。
assert validated  # 执行本行的状态、计算或校验逻辑。
assert not validate_index({"embed.weight": "missing"}, shards, expected_shapes)  # 执行本行的状态、计算或校验逻辑。
assert len(index) == 3  # 执行本行的状态、计算或校验逻辑。

## 4. 加载计划

先根据 index 构造去重且确定顺序的 shard 计划，再以受控内存预算逐 shard 校验和装载。plan 是可复放的控制面：它保存 bundle revision、目标并行拓扑和每个 shard 的状态；worker 不能自行猜测目录排序或跳过失败 shard。

In [ ]:
loaded_shapes = {"embed.weight": (8, 4), "layer.0.weight": (4, 4), "lm_head.weight": (8, 4)}  # 执行本行的状态、计算或校验逻辑。
def shape_errors(actual, expected):  # 执行本行的状态、计算或校验逻辑。
    return {key: (actual.get(key), shape) for key, shape in expected.items() if actual.get(key) != shape}  # 执行本行的状态、计算或校验逻辑。
assert shape_errors(loaded_shapes, expected_shapes) == {}  # 执行本行的状态、计算或校验逻辑。
assert "lm_head.weight" in shape_errors({"lm_head.weight": (7, 4)}, {"lm_head.weight": (8, 4)})  # 执行本行的状态、计算或校验逻辑。
assert shape_errors({"extra": (1,)}, {}) == {}  # 执行本行的状态、计算或校验逻辑。

## 5. 兼容性门禁

服务端还要核对 tokenizer/chat template、RoPE、KV head、dtype、adapter base revision 和 kernel 能力。模型能跑不代表输入接口正确：一个 special token 或模板漂移就会改变 token 序列、KV 前缀和线上行为，因此应在 admission 前整体拒绝。

In [ ]:
def build_plan(index_value):  # 执行本行的状态、计算或校验逻辑。
    return tuple(sorted(set(index_value.values())))  # 执行本行的状态、计算或校验逻辑。
plan = build_plan(index)  # 执行本行的状态、计算或校验逻辑。
assert plan == ("weights-01.safetensors", "weights-02.safetensors")  # 执行本行的状态、计算或校验逻辑。
assert len(plan) == 2  # 执行本行的状态、计算或校验逻辑。
assert build_plan({"a": "same", "b": "same"}) == ("same",)  # 执行本行的状态、计算或校验逻辑。

## 6. 原子发布与回滚

发布时先校验候选 manifest，再把 active pointer 从旧 revision 原子切到新 revision，并保留前一份可回滚制品。不要先覆盖目录再补 index；部分可见的 bundle 会让不同副本加载不同参数，造成难以复现的质量与安全问题。

In [ ]:
runtime = {"base": "tiny-1", "tokenizer": "tok-r3", "template": "chat-r2", "dtype": "bf16"}  # 执行本行的状态、计算或校验逻辑。
def admit(bundle_value, runtime_value, index_ok, shapes_ok):  # 执行本行的状态、计算或校验逻辑。
    return index_ok and shapes_ok and all(bundle_value[key] == runtime_value[key] for key in runtime_value)  # 执行本行的状态、计算或校验逻辑。
assert admit(bundle, runtime, True, True)  # 执行本行的状态、计算或校验逻辑。
assert not admit(bundle, {**runtime, "tokenizer": "tok-r4"}, True, True)  # 执行本行的状态、计算或校验逻辑。
assert not admit(bundle, runtime, False, True)  # 执行本行的状态、计算或校验逻辑。

## 7. 验收与边界

验收至少包含完整性、严格 key/shape、兼容矩阵、冷加载、生成 smoke test 和回滚演练。下面的小数据只证明元数据状态机；真实权重摘要、文件系统原子性、GPU 内存、分布式 barrier 与供应链签名仍需在生产环境额外验证。

In [ ]:
active_pointer = {"revision": "model-r6"}  # 执行本行的状态、计算或校验逻辑。
def publish(pointer, candidate, allowed):  # 执行本行的状态、计算或校验逻辑。
    return {"revision": candidate["revision"]} if allowed else dict(pointer)  # 执行本行的状态、计算或校验逻辑。
next_pointer = publish(active_pointer, bundle, admit(bundle, runtime, True, True))  # 执行本行的状态、计算或校验逻辑。
assert next_pointer["revision"] == "model-r7"  # 执行本行的状态、计算或校验逻辑。
assert publish(active_pointer, bundle, False)["revision"] == "model-r6"  # 执行本行的状态、计算或校验逻辑。
assert active_pointer["revision"] == "model-r6"  # 执行本行的状态、计算或校验逻辑。

## 8. 面试追问

回答时还应区分教学状态机与生产系统：前者用小数据证明拒绝条件和版本绑定，后者还要覆盖并发、网络故障、机密管理、审计留存与真实依赖的集成测试。任何无法由当前证据确认的状态，都应显式返回未验证、降级或人工升级，而不是由模型补全。

In [ ]:
load_report = {"revision": bundle["revision"], "plan": plan, "index_ok": validated, "shape_ok": not shape_errors(loaded_shapes, expected_shapes)}  # 执行本行的状态、计算或校验逻辑。
assert load_report["index_ok"] and load_report["shape_ok"]  # 执行本行的状态、计算或校验逻辑。
assert load_report["plan"][0] == "weights-01.safetensors"  # 执行本行的状态、计算或校验逻辑。
assert "sha" not in load_report  # 执行本行的状态、计算或校验逻辑。

## 面试总结

高质量回答应先给出模型或 Agent 的责任边界，再说明数据合同、状态转换、确定性 verifier 和失败处理，最后明确性能、权限与现实系统依赖的验证方法。不要把一次函数返回、模型文本或受控小样本断言误称为线上正确性。